In [1]:
import pandas as pd
import requests
from io import StringIO

In [2]:
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [3]:
url = "https://www3.cde.ca.gov/demo-downloads/attendance/chronicabsenteeism25.txt"

# Getting the txt file info
r = requests.get(url)
r.raise_for_status()
text = r.content.decode("cp1252", errors="replace")  # or "latin-1"

In [4]:
# Converting the text into pandas df
df = pd.read_csv(StringIO(text), sep="\t")
df.columns = df.columns.str.strip()

In [5]:
# Filtering the DataFrame to only show school-level data for grades 9-12 (high school) and removing all District Office locations (school code == 0.0)
# Also cleaning up the columns to keep only those we want
df = df[(df['Aggregate Level'] == 'S') & (df['Reporting Category'] == 'GR912') & (df['School Code'] != 0.0)]
df = df.drop(columns = [col for col in df.columns if col in ['Academic Year', 'Aggregate Level', 'County Name', 'District Name', 'Reporting Category']])
df['School Code'] = df['School Code'].apply(lambda code: f"{int(code):07d}")
df = df.reset_index(drop=True)

In [6]:
df = df[df['School Code'].isin(school_codes_list)]
df = df.drop(columns = ['ChronicAbsenteeismEligibleCumulativeEnrollment', 'ChronicAbsenteeismCount'])

In [7]:
df

,County Code,District Code,School Code,School Name,Charter School,DASS,ChronicAbsenteeismRate
4,1,10017.0,0130625,Alternatives in Action,Yes,Yes,17.2
7,1,61119.0,0106401,Alameda Science and Technology Institute,No,No,8.0
9,1,61119.0,0130229,Alameda High,No,No,10.9
13,1,61127.0,0130450,Albany High,No,No,18.3
15,1,61143.0,0131177,Berkeley High,No,No,13.5
...,...,...,...,...,...,...,...
2660,57,72710.0,5738802,Woodland Senior High,No,No,11.9
2664,58,10587.0,5830047,Harry P B Carden,No,Yes,0.0
2669,58,72736.0,5830013,Lindhurst High,No,No,24.8
2672,58,72736.0,5835202,Marysville High,No,No,20.4


In [8]:
df['School Code'].nunique()

1396

In [9]:
df.to_csv('../cleaned_data/absenteeism_data.csv')